# Lab 9. SARIMA and Multiple Seasonality

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-09-sarima-multiple-seasonality-lab.ipynb)

This lab is designed for **independent study**. It includes background before programming, guided code, interpretation checkpoints, and short exercises. You should be able to read the explanations, run the code, and modify the examples on your own.

## Learning goals

By the end of this lab, you should be able to:

1. Explain why ordinary ARIMA models may fail for seasonal time series.
2. Construct seasonal lag and seasonal difference features.
3. Simulate pure seasonal AR and pure seasonal MA processes.
4. Recognize seasonal dependence from ACF and PACF plots.
5. Fit SARIMA models with `statsmodels`.
6. Compare seasonal naive, ARIMA, and SARIMA forecasts.
7. Use residual diagnostics and Ljung-Box tests after fitting a seasonal model.
8. Understand how multiple seasonality can be handled using Fourier features and dynamic regression.
9. Use AI tools to critique a seasonal forecasting workflow without blindly trusting the output.

## Google Colab math note

This notebook uses simple LaTeX notation that displays reliably in Google Colab.

## 1. Background: why seasonality needs special treatment

A time series is **seasonal** when observations separated by a fixed period tend to be related. Examples:

- Monthly retail sales often repeat every 12 months.
- Hourly electricity demand often repeats every 24 hours.
- Daily web traffic often repeats every 7 days.
- Minute-level sensor data may contain several periodic patterns at the same time.

If the seasonal period is $s$, then the seasonal lag operator is $B^s$:

$$
B^s X_t = X_{t-s}.
$$

The seasonal difference is

$$

abla_s X_t = X_t - X_{t-s} = (1-B^s)X_t.
$$

For monthly data, $s=12$, so

$$

abla_{12}X_t = X_t - X_{t-12}.
$$

This compares each month with the same month one year earlier.

A nonseasonal ARIMA model is written as ARIMA($p,d,q$). A seasonal model is often written as

$$
	ext{SARIMA}(p,d,q)(P,D,Q)_s.
$$

Here:

- $p,d,q$ describe short-term nonseasonal ARIMA behavior.
- $P,D,Q$ describe seasonal AR, seasonal differencing, and seasonal MA behavior.
- $s$ is the seasonal period.

The most important practical point is this:

> Do not fit a complicated SARIMA model first. Start with plots, seasonal differences, ACF/PACF, and simple baselines.

## 2. Setup

Run this cell first. It imports packages and defines helper functions used throughout the lab.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

rng = np.random.default_rng(7339)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True


def plot_series(y, title="Time series", ylabel="value"):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(np.arange(len(y)), y)
    ax.set_title(title)
    ax.set_xlabel("time")
    ax.set_ylabel(ylabel)
    plt.show()


def seasonal_difference(y, s):
    y = np.asarray(y, dtype=float)
    return y[s:] - y[:-s]


def ordinary_difference(y, d=1):
    out = np.asarray(y, dtype=float)
    for _ in range(d):
        out = np.diff(out)
    return out


def train_test_split_series(y, h):
    y = np.asarray(y, dtype=float)
    return y[:-h], y[-h:]


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


def seasonal_naive_forecast(train, h, s):
    # Repeat the last observed season until h forecasts are produced.
    train = np.asarray(train, dtype=float)
    last_season = train[-s:]
    reps = int(np.ceil(h / s))
    return np.tile(last_season, reps)[:h]


def diagnostic_plots(residuals, max_lag=36, title_prefix="Residual"):
    residuals = pd.Series(residuals).dropna().to_numpy()
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(residuals)
    ax.axhline(0, linewidth=1)
    ax.set_title(f"{title_prefix} time plot")
    ax.set_xlabel("time")
    ax.set_ylabel("residual")
    plt.show()

    fig, ax = plt.subplots(figsize=(10, 3))
    plot_acf(residuals, lags=max_lag, ax=ax)
    ax.set_title(f"{title_prefix} ACF")
    plt.show()


def adf_summary(y, name="series"):
    stat, pvalue, used_lag, nobs, crit, icbest = adfuller(np.asarray(y), autolag="AIC")
    print(f"ADF test for {name}")
    print(f"  test statistic: {stat: .3f}")
    print(f"  p-value:        {pvalue: .4f}")
    print(f"  used lags:      {used_lag}")
    print(f"  observations:   {nobs}")
    print("  rough interpretation:", "likely stationary" if pvalue < 0.05 else "unit-root behavior not rejected")

## 3. Simulate a monthly seasonal series

Before using real data, it is useful to create a controlled example. We simulate a monthly series with:

1. a slowly increasing trend,
2. a yearly seasonal pattern with period $s=12$,
3. correlated short-term noise.

The model is not exactly SARIMA. That is intentional. Real data are rarely exactly SARIMA. Our goal is to learn a practical modeling workflow.

In [ ]:
n_years = 16
s = 12
n = n_years * s

t = np.arange(n)
trend = 0.08 * t
seasonal = 8 * np.sin(2 * np.pi * t / s) + 3 * np.cos(2 * np.pi * t / s)

# Correlated short-term noise: AR(1)-type disturbance.
noise = np.zeros(n)
innov = rng.normal(0, 1.2, size=n)
for i in range(1, n):
    noise[i] = 0.55 * noise[i - 1] + innov[i]

x = 60 + trend + seasonal + noise

monthly_index = pd.date_range(start="2010-01-01", periods=n, freq="MS")
series = pd.Series(x, index=monthly_index, name="synthetic_monthly")

series.head()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
series.plot(ax=ax)
ax.set_title("Synthetic monthly series: trend + yearly seasonality + correlated noise")
ax.set_xlabel("date")
ax.set_ylabel("value")
plt.show()

### Checkpoint 1

Answer before moving on:

1. What visual evidence suggests this series is not stationary?
2. What visual evidence suggests the period is 12?
3. Why might an ordinary ARIMA model miss part of the structure?

## 4. Seasonal plots: comparing the same month across years

A useful EDA tool for monthly data is a seasonal plot. Instead of plotting time on one long axis, we plot each year as a curve over months 1 through 12.

If curves have similar shape each year, there is evidence of yearly seasonality.

In [ ]:
seasonal_df = pd.DataFrame({
    "year": series.index.year,
    "month": series.index.month,
    "value": series.values
})

fig, ax = plt.subplots(figsize=(10, 5))
for year, group in seasonal_df.groupby("year"):
    ax.plot(group["month"], group["value"], marker="o", alpha=0.65, label=str(year))
ax.set_title("Seasonal plot: one curve per year")
ax.set_xlabel("month")
ax.set_ylabel("value")
ax.set_xticks(range(1, 13))
ax.legend(ncol=4, fontsize=8)
plt.show()

## 5. Seasonal differencing

The seasonal difference compares each observation to the observation one season earlier:

$$

abla_s X_t = X_t - X_{t-s}.
$$

For monthly data with yearly seasonality, use $s=12$.

A seasonal difference is not the same as an ordinary first difference:

$$

abla X_t = X_t - X_{t-1}.
$$

The seasonal difference removes repeated annual patterns. The ordinary difference removes local trend.

In [ ]:
dx_ordinary = ordinary_difference(series.values, d=1)
dx_seasonal = seasonal_difference(series.values, s=12)
dx_both = ordinary_difference(dx_seasonal, d=1)

plot_series(series.values, title="Original monthly series")
plot_series(dx_ordinary, title="Ordinary first difference: X[t] - X[t-1]", ylabel="change")
plot_series(dx_seasonal, title="Seasonal difference: X[t] - X[t-12]", ylabel="seasonal change")
plot_series(dx_both, title="Seasonal difference followed by ordinary difference", ylabel="change")

In [ ]:
adf_summary(series.values, name="original series")
print()
adf_summary(dx_seasonal, name="seasonally differenced series")
print()
adf_summary(dx_both, name="seasonally and ordinarily differenced series")

### Checkpoint 2

The ADF test is only one diagnostic. It should not replace plots and domain reasoning.

1. Which transformed series looks most stable?
2. Does seasonal differencing alone remove the trend completely?
3. What could go wrong if we difference too many times?

## 6. ACF and PACF for seasonal identification

Seasonality often appears as large ACF values at lags $s, 2s, 3s, \ldots$.

For monthly data, check lags 12, 24, 36, and so on.

Rules of thumb:

- A seasonal MA(1) component often gives a strong ACF spike near lag $s$.
- A seasonal AR(1) component often gives seasonal decay at lags $s, 2s, 3s, \ldots$.
- The ACF and PACF must be interpreted together.
- Do not choose a model only from one plot.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
plot_acf(series.values, lags=48, ax=axes[0])
axes[0].set_title("ACF of original series")
plot_acf(dx_seasonal, lags=48, ax=axes[1])
axes[1].set_title("ACF after seasonal differencing")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
plot_pacf(dx_seasonal, lags=36, ax=axes[0], method="ywm")
axes[0].set_title("PACF after seasonal differencing")
plot_acf(dx_both, lags=36, ax=axes[1])
axes[1].set_title("ACF after seasonal + ordinary differencing")
plt.tight_layout()
plt.show()

## 7. Simulate pure seasonal AR and seasonal MA processes

A pure seasonal AR(1) model with period $s$ has the form

$$
X_t = \Phi X_{t-s} + W_t.
$$

A pure seasonal MA(1) model with period $s$ has the form

$$
X_t = W_t + \Theta W_{t-s}.
$$

These models help us learn what seasonal dependence looks like in plots.

In [ ]:
def simulate_seasonal_ar1(n, s=12, Phi=0.75, sigma=1.0, seed=1):
    local_rng = np.random.default_rng(seed)
    w = local_rng.normal(0, sigma, size=n)
    x = np.zeros(n)
    for t in range(n):
        if t >= s:
            x[t] = Phi * x[t - s] + w[t]
        else:
            x[t] = w[t]
    return x


def simulate_seasonal_ma1(n, s=12, Theta=0.75, sigma=1.0, seed=2):
    local_rng = np.random.default_rng(seed)
    w = local_rng.normal(0, sigma, size=n)
    x = np.zeros(n)
    for t in range(n):
        if t >= s:
            x[t] = w[t] + Theta * w[t - s]
        else:
            x[t] = w[t]
    return x

x_sar = simulate_seasonal_ar1(240, s=12, Phi=0.75, seed=10)
x_sma = simulate_seasonal_ma1(240, s=12, Theta=0.75, seed=20)

plot_series(x_sar, title="Pure seasonal AR(1) with period 12")
plot_series(x_sma, title="Pure seasonal MA(1) with period 12")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
plot_acf(x_sar, lags=60, ax=axes[0])
axes[0].set_title("ACF: pure seasonal AR(1)")
plot_acf(x_sma, lags=60, ax=axes[1])
axes[1].set_title("ACF: pure seasonal MA(1)")
plt.tight_layout()
plt.show()

### Checkpoint 3

Look at the ACF plots.

1. Which process has seasonal autocorrelation that decays across seasonal lags?
2. Which process has a stronger cutoff-type behavior near lag 12?
3. Why are these only rules of thumb for real data?

## 8. Forecast baselines: seasonal naive method

Before fitting SARIMA, always build a simple baseline.

For seasonal data, the seasonal naive forecast is:

$$
\hat X_{t+h} = X_{t+h-s}.
$$

For monthly data, this says: forecast next January using last January, next February using last February, and so on.

A SARIMA model is only useful if it improves on simple baselines in a time-aware evaluation.

In [ ]:
h = 24
train, test = train_test_split_series(series.values, h=h)

seasonal_naive_pred = seasonal_naive_forecast(train, h=h, s=12)

print("Seasonal naive RMSE:", round(rmse(test, seasonal_naive_pred), 3))
print("Seasonal naive MAE: ", round(mae(test, seasonal_naive_pred), 3))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(np.arange(len(series)), series.values, label="observed")
forecast_index = np.arange(len(train), len(train) + h)
ax.plot(forecast_index, seasonal_naive_pred, marker="o", label="seasonal naive forecast")
ax.axvline(len(train) - 1, linestyle="--", label="train/test split")
ax.set_title("Seasonal naive forecast")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()
plt.show()

## 9. Fit a nonseasonal ARIMA model

We first fit a nonseasonal ARIMA model. This is a useful comparison because it shows what happens when the model ignores seasonality.

We fit the model only on the training set and evaluate on the final 24 months.

In [ ]:
arima_model = ARIMA(train, order=(1, 1, 1))
arima_res = arima_model.fit()

arima_forecast = arima_res.get_forecast(steps=h)
arima_pred = arima_forecast.predicted_mean

print(arima_res.summary())
print()
print("ARIMA(1,1,1) RMSE:", round(rmse(test, arima_pred), 3))
print("ARIMA(1,1,1) MAE: ", round(mae(test, arima_pred), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(np.arange(len(series)), series.values, label="observed")
ax.plot(forecast_index, seasonal_naive_pred, marker="o", label="seasonal naive")
ax.plot(forecast_index, arima_pred, marker="o", label="ARIMA(1,1,1)")
ax.axvline(len(train) - 1, linestyle="--", label="train/test split")
ax.set_title("Nonseasonal ARIMA compared with seasonal naive")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()
plt.show()

## 10. Fit a SARIMA model

Now we fit a seasonal model. A common starting point for monthly seasonal data is a model such as

$$
	ext{SARIMA}(1,0,1)(0,1,1)_{12}.
$$

This means:

- short-term ARMA(1,1) behavior,
- one seasonal difference at lag 12,
- one seasonal MA term at lag 12.

This is not automatically the best model. It is a reasonable first candidate.

In [ ]:
sarima_model = SARIMAX(
    train,
    order=(1, 0, 1),
    seasonal_order=(0, 1, 1, 12),
    trend="c",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarima_res = sarima_model.fit(disp=False, maxiter=100)

sarima_forecast = sarima_res.get_forecast(steps=h)
sarima_pred = sarima_forecast.predicted_mean
sarima_ci = sarima_forecast.conf_int()

print(sarima_res.summary())
print()
print("SARIMA(1,0,1)(0,1,1)[12] RMSE:", round(rmse(test, sarima_pred), 3))
print("SARIMA(1,0,1)(0,1,1)[12] MAE: ", round(mae(test, sarima_pred), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(np.arange(len(series)), series.values, label="observed")
ax.plot(forecast_index, seasonal_naive_pred, marker="o", label="seasonal naive")
ax.plot(forecast_index, arima_pred, marker="o", label="ARIMA(1,1,1)")
ax.plot(forecast_index, sarima_pred, marker="o", label="SARIMA")
ax.fill_between(
    forecast_index,
    sarima_ci[:, 0],
    sarima_ci[:, 1],
    alpha=0.2,
    label="SARIMA 95% interval",
)
ax.axvline(len(train) - 1, linestyle="--", label="train/test split")
ax.set_title("Forecast comparison")
ax.set_xlabel("time")
ax.set_ylabel("value")
ax.legend()
plt.show()

In [ ]:
comparison = pd.DataFrame({
    "model": ["seasonal naive", "ARIMA(1,1,1)", "SARIMA(1,0,1)(0,1,1)[12]"],
    "RMSE": [
        rmse(test, seasonal_naive_pred),
        rmse(test, arima_pred),
        rmse(test, sarima_pred),
    ],
    "MAE": [
        mae(test, seasonal_naive_pred),
        mae(test, arima_pred),
        mae(test, sarima_pred),
    ],
})

comparison.round(3)

### Checkpoint 4

1. Which model performs best on RMSE?
2. Which model performs best on MAE?
3. Does a more complicated model always perform better?
4. Why should model comparison be done on held-out future data rather than randomly shuffled data?

## 11. Residual diagnostics for the SARIMA model

After fitting a model, the residuals should look like white noise. We check:

1. residual time plot,
2. residual ACF,
3. Ljung-Box test.

The Ljung-Box test checks whether residual autocorrelations up to a specified lag are jointly small. A small p-value suggests remaining autocorrelation.

In [ ]:
sarima_residuals = sarima_res.resid

diagnostic_plots(sarima_residuals, max_lag=36, title_prefix="SARIMA residual")

lb = acorr_ljungbox(sarima_residuals, lags=[12, 24, 36], return_df=True)
lb

### Checkpoint 5

1. Do the residuals look centered near zero?
2. Are there large residual ACF spikes at seasonal lags?
3. What do the Ljung-Box p-values suggest?
4. If the residuals still show seasonal dependence, what model changes might you try?

## 12. Small SARIMA model search

In practice, we often compare a small set of candidate models. We should not blindly search over many models without thinking. But a small grid can be useful.

Here we compare a few SARIMA models using AIC and held-out forecast error.

In [ ]:
candidates = [
    ((0, 0, 1), (0, 1, 1, 12)),
    ((1, 0, 0), (0, 1, 1, 12)),
    ((1, 0, 1), (0, 1, 1, 12)),
    ((1, 0, 1), (1, 1, 0, 12)),
    ((0, 1, 1), (0, 1, 1, 12)),
]

rows = []

for order, seasonal_order in candidates:
    try:
        model = SARIMAX(
            train,
            order=order,
            seasonal_order=seasonal_order,
            trend="c",
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = model.fit(disp=False, maxiter=100)
        pred = res.get_forecast(steps=h).predicted_mean
        rows.append({
            "order": str(order),
            "seasonal_order": str(seasonal_order),
            "AIC": res.aic,
            "BIC": res.bic,
            "test_RMSE": rmse(test, pred),
            "test_MAE": mae(test, pred),
        })
    except Exception as e:
        rows.append({
            "order": str(order),
            "seasonal_order": str(seasonal_order),
            "AIC": np.nan,
            "BIC": np.nan,
            "test_RMSE": np.nan,
            "test_MAE": np.nan,
        })

model_table = pd.DataFrame(rows).sort_values("test_RMSE")
model_table.round(3)

### Checkpoint 6

1. Does the lowest AIC model also have the lowest test RMSE?
2. Which criterion would you emphasize for a forecasting task: AIC or held-out test error?
3. Why is it dangerous to repeatedly tune a model on the test set?

## 13. Multiple seasonality with Fourier features

Some time series have more than one seasonal period. For example, hourly electricity demand may have:

- daily seasonality with period 24,
- weekly seasonality with period 168.

A full seasonal ARIMA model with multiple seasonal periods can become difficult. One common alternative is to use Fourier features as regressors:

$$
\sin(2\pi kt / s), \quad \cos(2\pi kt / s).
$$

These features allow a regression or ARIMA-with-exogenous-variables model to represent smooth seasonal patterns.

Here we simulate a series with two seasonal periods and fit a simple regression using Fourier features.

In [ ]:
def fourier_features(t, period, K, prefix):
    # Create sine and cosine Fourier features for a given period.
    data = {}
    for k in range(1, K + 1):
        data[f"{prefix}_sin_{k}"] = np.sin(2 * np.pi * k * t / period)
        data[f"{prefix}_cos_{k}"] = np.cos(2 * np.pi * k * t / period)
    return pd.DataFrame(data)

n_hourly = 24 * 35
th = np.arange(n_hourly)

daily = 4.0 * np.sin(2 * np.pi * th / 24) + 1.5 * np.cos(2 * np.pi * th / 24)
weekly = 2.5 * np.sin(2 * np.pi * th / 168)
trend_hourly = 0.003 * th
noise_hourly = rng.normal(0, 0.8, size=n_hourly)

y_hourly = 20 + trend_hourly + daily + weekly + noise_hourly

plot_series(y_hourly, title="Simulated hourly series with daily and weekly seasonality")

In [ ]:
X_daily = fourier_features(th, period=24, K=2, prefix="daily")
X_weekly = fourier_features(th, period=168, K=2, prefix="weekly")
X = pd.concat([pd.Series(th, name="trend"), X_daily, X_weekly], axis=1)
X.insert(0, "intercept", 1.0)

# Least squares fit: beta_hat = (X^T X)^(-1) X^T y
X_mat = X.to_numpy()
y_vec = y_hourly
beta_hat, *_ = np.linalg.lstsq(X_mat, y_vec, rcond=None)
fitted = X_mat @ beta_hat
resid = y_vec - fitted

plot_series(y_hourly, title="Observed hourly series")
plot_series(fitted, title="Fitted trend + Fourier seasonal components")
plot_series(resid, title="Residual after removing Fourier seasonal features", ylabel="residual")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7))
plot_acf(y_hourly, lags=200, ax=axes[0])
axes[0].set_title("ACF before Fourier seasonal regression")
plot_acf(resid, lags=200, ax=axes[1])
axes[1].set_title("ACF after Fourier seasonal regression")
plt.tight_layout()
plt.show()

### Checkpoint 7

1. Which seasonal periods are visible in the original ACF?
2. Does the Fourier regression remove all dependence?
3. Why might we combine Fourier features with ARMA errors?

## 14. AI-assisted study prompt

Use an AI assistant only after you have looked at the plots and results yourself.

Copy and adapt this prompt:

> I am analyzing a monthly time series with possible yearly seasonality. I used a seasonal plot, seasonal differencing with period 12, ACF/PACF plots, a seasonal naive baseline, and SARIMA candidate models. Here are my plots and the model comparison table. Please critique my workflow. Focus on possible over-differencing, remaining residual autocorrelation, leakage in train/test splitting, and whether the SARIMA model really improves over seasonal naive forecasting. Do not invent conclusions not supported by the results.

After the AI responds, write your own verification notes:

1. Which parts of the response are directly supported by plots or numbers?
2. Which parts are guesses?
3. What additional diagnostic would you run next?

## 15. Mini-project

Choose one of the following.

### Option A: monthly seasonal data

Use the synthetic monthly series from this lab or load another monthly series. Build a forecasting report that includes:

1. time plot,
2. seasonal plot,
3. ordinary and seasonal differencing,
4. ACF/PACF diagnostics,
5. seasonal naive baseline,
6. at least two SARIMA candidates,
7. residual diagnostics,
8. final recommendation.

### Option B: multiple seasonality

Simulate or load hourly data with at least two seasonal periods. Build:

1. Fourier features for each period,
2. a regression model with Fourier features,
3. residual ACF diagnostics,
4. a short discussion of what dependence remains.

### Option C: model comparison

Compare seasonal naive, nonseasonal ARIMA, SARIMA, and Fourier regression on the same data. Use a chronological train/test split and report RMSE and MAE.

## 16. Lab checklist

Before leaving this lab, make sure you can do the following without looking at the solution:

- Explain the difference between ordinary differencing and seasonal differencing.
- Compute $X_t-X_{t-12}$ for monthly data.
- Explain why ACF spikes at lags 12, 24, and 36 suggest yearly seasonality.
- Simulate pure seasonal AR and seasonal MA processes.
- Fit a SARIMA model using `statsmodels`.
- Compare SARIMA against a seasonal naive baseline.
- Check residuals using residual plots, ACF, and Ljung-Box tests.
- Explain why Fourier features are useful for multiple seasonality.
- Identify at least two ways seasonal modeling can go wrong.